In [ ]:
from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F
import math
import matplotlib.pyplot as plt
import seaborn as sns
import random
import os
import time



class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                     .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()

        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)

        # manual implementation of attention
        y = F.scaled_dot_product_attention(q,k,v, is_causal=True)

        y = y.transpose(1, 2).contiguous().view(B, T, C)

        # output projection
        y = self.c_proj(y)
        return y



class MLP(nn.Module):
  def __init__(self,config):
    super().__init__()
    self.c_fc = nn.Linear(config.n_embd, 4*config.n_embd)
    self.gelu = nn.GELU(approximate='tanh')
    self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)
    self.c_proj.NANOGPT_SCALE_INIT = 1

  def forward(self,x):
    x = self.c_fc(x)
    x = self.gelu(x)
    x = self.c_proj(x)
    return x


class Block(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.ln_1 = nn.LayerNorm(config.n_embd)
    self.attn = CausalSelfAttention(config)
    self.ln_2 = nn.LayerNorm(config.n_embd)
    self.mlp = MLP(config)

  def forward(self, x):
    x = x + self.attn(self.ln_1(x))
    x = x + self.mlp(self.ln_2(x))
    return x



@dataclass
class GrokkingConfig:
    vocab_size = 16
    block_size = 16
    n_layer = 4
    n_head = 4
    n_embd = 128




grad_accum_steps = 32




class GPT(nn.Module):

  def __init__(self, config):
    super().__init__()
    self.config = config

    self.transformer = nn.ModuleDict(dict(
      wte = nn.Embedding(config.vocab_size, config.n_embd),
      wpe = nn.Embedding(config.block_size, config.n_embd),
      h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
      ln_f = nn.LayerNorm(config.n_embd),
    ))
    self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

    self.transformer.wte.weight = self.lm_head.weight

    self.apply(self._init_weights)

  def _init_weights(self, module):
    if isinstance(module, nn.Linear):
      std = 0.02
      if hasattr(module, 'NANOGPT_SCALE_INIT'):
        std *= (2 * self.config.n_layer) ** -0.5
      torch.nn.init.normal_(module.weight, mean = 0.0, std = std)
      if module.bias is not None:
        torch.nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
      torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)


  def forward(self,idx, targets=None):
    B,T = idx.size()
    assert T <= self.config.block_size
    pos = torch.arange(0, T, dtype = torch.long, device=idx.device)
    pos_emb = self.transformer.wpe(pos)
    tok_emb = self.transformer.wte(idx)
    x = tok_emb + pos_emb
    for block in self.transformer.h:
      x = block(x)

    x = self.transformer.ln_f(x)
    logits = self.lm_head(x)
    loss = None
    if targets is not None:
      loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
    return logits, loss

In [ ]:
chars = list("0123456789+=\n")
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
    return [stoi[c] for c in s]

def decode(l):
    return ''.join([itos[i] for i in l])

vocab_size = len(chars)
print(f"Vocab size: {vocab_size}")

class MathDataLoader:
    def __init__(self, filename, B, T):
        self.B = B
        self.T = T

        with open(filename, 'r') as f:
            data = f.read()

        tokens = encode(data)
        self.data = torch.tensor(tokens, dtype=torch.long)
        print(f"[{filename}] Downloaded {len(self.data):,} tokens")

    def next_batch(self):
        ix = torch.randint(len(self.data) - self.T, (self.B,))

        x = torch.stack([self.data[i : i + self.T] for i in ix])
        y = torch.stack([self.data[i + 1 : i + self.T + 1] for i in ix])

        return x, y

In [ ]:
equations = []
for a in range(100):
    for b in range(100):
        c = a + b
        eq = f"{a:02d}+{b:02d}={c:03d}"
        equations.append(eq)

random.seed(42)
random.shuffle(equations)

split_idx = int(0.8 * len(equations))
train_data = equations[:split_idx]
val_data = equations[split_idx:]

with open('train_math.txt', 'w') as f:
    f.write('\n'.join(train_data) + '\n')
with open('val_math.txt', 'w') as f:
    f.write('\n'.join(val_data) + '\n')

print(f"Train: {len(train_data)} problems. Val: {len(val_data)} problems")


In [ ]:
model = GPT(GrokkingConfig())
model.load_state_dict(torch.load('grokking_addition_15k_partial_carry.pt', map_location=device, weights_only=True))
model.to(device)
model.eval()

train_loader = MathDataLoader('train_math_rev.txt', B=128, T=16)
val_loader = MathDataLoader('val_math_rev.txt', B=128, T=16)

model = GPT(GrokkingConfig())

model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1.0, betas=(0.9, 0.98))

max_steps = 15000




for step in range(max_steps):
  t0 = time.time()
  if step % 250 == 0 or step == max_steps - 1:
    model.eval()
    val_loss_steps = 20

    with torch.no_grad():
            for _ in range(val_loss_steps):
                x_val, y_val = val_loader.next_batch()
                x_val, y_val = x_val.to(device), y_val.to(device)
                val_logits, val_loss = model(x_val, y_val)

    print(f"Validation step {step:4d} | val_loss: {val_loss:.4f}")
    model.train()

  x, y = train_loader.next_batch()
  x, y = x.to(device), y.to(device)
  logits, loss = model(x,y)

  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  torch.cuda.synchronize()
  t1 = time.time()
  dt = (t1 - t0) * 1000
  tokens_processed = train_loader.B * train_loader.T * grad_accum_steps
  tokens_per_sec = tokens_processed / (t1 - t0)

  print(f"step {step:4d} | loss: {loss.item():.4f} | dt: {dt:.2f}ms | tok/sec: {tokens_per_sec:.2f}")


In [ ]:
model.eval()

equation = "35+25="
tokens = encode(equation)
x_test = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0)

max_new_tokens = 4

with torch.no_grad():
    for _ in range(max_new_tokens):
        x_cond = x_test[:, -GrokkingConfig.block_size:]
        logits, _ = model(x_cond)

        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)

        idx_next = torch.argmax(probs, dim=-1, keepdim=True)
        x_test = torch.cat((x_test, idx_next), dim=1)

result = decode(x_test[0].tolist())
print(f"Result: {result}")

In [ ]:
text = "15+15=020"
tokens = encode(text)
T_len = len(tokens)
x = torch.tensor([tokens], dtype=torch.long, device=device)
labels = list(text)

with torch.no_grad():
    pos = torch.arange(0, T_len, dtype=torch.long, device=device)
    x_stream = model.transformer.wte(x) + model.transformer.wpe(pos)

    fig, axes = plt.subplots(4, 4, figsize=(16, 16))
    fig.suptitle("15+15=020", fontsize=20)

    for layer_idx in range(model.config.n_layer):

        x_ln = model.transformer.h[layer_idx].ln_1(x_stream)
        qkv = model.transformer.h[layer_idx].attn.c_attn(x_ln)
        head_size = model.config.n_embd // model.config.n_head

        q, k, v = qkv.split(model.config.n_embd, dim=2)
        q = q.view(1, T_len, model.config.n_head, head_size).transpose(1,2)
        k = k.view(1, T_len, model.config.n_head, head_size).transpose(1,2)


        att = q @ k.transpose(-1,-2) * (1.0 / math.sqrt(head_size))

        tril = torch.tril(torch.ones(T_len, T_len, device=device))
        mask = tril.view(1, 1, T_len, T_len)
        att = att.masked_fill(mask == 0, float('-inf'))

        probs = F.softmax(att, dim=-1)

        for head_idx in range(model.config.n_head):
            ax = axes[layer_idx, head_idx]
            matrix_to_plot = probs[0, head_idx].cpu().numpy()

            sns.heatmap(matrix_to_plot, xticklabels=labels, yticklabels=labels,
                        cmap='Reds', annot=True, fmt=".2f", square=True,
                        cbar=False, ax=ax)
            ax.set_title(f"L{layer_idx} H{head_idx}")

plt.tight_layout()
file_name = 'math_xray_15_plus_15.png'
plt.savefig(file_name, dpi=300, bbox_inches='tight')
plt.show()
from google.colab import files
files.download(file_name)


In [ ]:




equations = []
for a in range(100):
    for b in range(100):
        c = a + b
        str_a = f"{a:02d}"
        str_b = f"{b:02d}"
        str_c = f"{c:03d}"
        eq = f"{str_a}+{str_b}={str_c[::-1]}"
        equations.append(eq)

random.seed(42)
random.shuffle(equations)

split_idx = int(0.8 * len(equations))
train_data = equations[:split_idx]
val_data = equations[split_idx:]

with open('train_math_rev.txt', 'w') as f:
    f.write('\n'.join(train_data) + '\n')
with open('val_math_rev.txt', 'w') as f:
    f.write('\n'.join(val_data) + '\n')


train_loader = MathDataLoader('train_math_rev.txt', B=128, T=16)
val_loader = MathDataLoader('val_math_rev.txt', B=128, T=16)

model = GPT(GrokkingConfig())
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1.0, betas=(0.9, 0.98))

max_steps = 15000


for step in range(max_steps):
    x, y = train_loader.next_batch()
    x, y = x.to(device), y.to(device)

    optimizer.zero_grad(set_to_none=True)
    logits, loss = model(x, y)
    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    if step % 250 == 0 or step == max_steps - 1:
        model.eval()
        with torch.no_grad():
            x_val, y_val = val_loader.next_batch()
            x_val, y_val = x_val.to(device), y_val.to(device)
            _, val_loss = model(x_val, y_val)
        print(f"step {step:5d} | train_loss: {loss.item():.4f} | val_loss: {val_loss.item():.4f}")
        model.train()

In [ ]:
for param_group in optimizer.param_groups:
    param_group['lr'] = 3e-4

mega_steps = 100000

for step in range(mega_steps):
    x, y = train_loader.next_batch()
    x, y = x.to(device), y.to(device)

    optimizer.zero_grad(set_to_none=True)
    logits, loss = model(x, y)
    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    if step % 5000 == 0 or step == mega_steps - 1:
        model.eval()
        with torch.no_grad():
            x_val, y_val = val_loader.next_batch()
            x_val, y_val = x_val.to(device), y_val.to(device)
            _, val_loss = model(x_val, y_val)
        print(f"Mega step {step:6d} | train_loss: {loss.item():.4f} | val_loss: {val_loss.item():.4f}")
        model.train()


In [ ]:

model.eval()

test_equations = [
    "14+35=",
    "15+15=",
    "04+37=",
    "99+99="
]

with torch.no_grad():
    for eq in test_equations:
        tokens = encode(eq)
        x_test = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0)


        for _ in range(3):
            x_cond = x_test[:, -GrokkingConfig.block_size:]
            logits, _ = model(x_cond)
            probs = F.softmax(logits[:, -1, :], dim=-1)

            idx_next = torch.argmax(probs, dim=-1, keepdim=True)
            x_test = torch.cat((x_test, idx_next), dim=1)

        raw_result = decode(x_test[0].tolist())

        prompt_part, ans_part = raw_result.split('=')
        real_math_answer = answer_part[::-1]

        print("-" * 30)
        print(f"Prompt: {prompt_part}=")
        print(f"Reverse: {answer_part}")
        print(f"Normal view: {prompt_part}={real_math_answer}")